In [ ]:
# ============================================================
# Cell 0: Setup — load config, sample IDs, embeddings (303), vocab
# ============================================================
import os, sys, json, re, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm
from omegaconf import OmegaConf
from transformers import BertTokenizer
from torch.utils.data import DataLoader

os.environ['CUDA_VISIBLE_DEVICES'] = '6'
sys.path.append('/home/yancui/Haiku/src')

from models import Haiku, MarkerEmbedding
from data import (TrimodalDatasetViTEmbedding, TrimodalDatasetViTPickeVerion,
                  custom_collate_fn_embedding)
from utils import PerChannelSelfStandardization, CustomGaussianBlurTorch

import warnings
warnings.filterwarnings('ignore')

cfg = OmegaConf.load('/home/yancui/Haiku/src/configs/config.yaml')

# ---- Sample IDs (same as tri-retrival.ipynb / biomarker_infer_metadata_token) ----
sample_dict = json.load(open('/home/yancui/Haiku/src/training/overlap_samples_final.json'))
sample_ids = list(sample_dict.keys())
with open('/home/yancui/OmicsAnnotator/test_regions.txt', 'r') as f:
    test_ids = [line.strip() for line in f if line.strip()]
holdout_list = pd.read_csv('/home/yancui/tier2_acquisition_ids_huanglab (3).csv')['ACQUISITION_ID'].tolist()
overlap_holdout = list(set(holdout_list) & set(sample_ids))
new_holdout = list(json.load(open('/home/yancui/Haiku/preprocessing/overlap_samples_new.json')).keys())
sample_ids = list(set(test_ids + list(set(overlap_holdout) - set(new_holdout))))
ref_ids = sorted(sample_ids)
print(f'ref_ids: {len(ref_ids)} regions')

# ---- Pre-computed embeddings (res_embedding_303 — latest) ----
emb_dir = '/home/yancui/Haiku/outputs/multimodal_embeddings'
he_embedding      = torch.load(f'{emb_dir}/he_embedding.pt', map_location='cpu')
codex_embedding   = torch.load(f'{emb_dir}/codex_embedding.pt', map_location='cpu')
text_embedding    = torch.load(f'{emb_dir}/text_embedding.pt', map_location='cpu')
region_label      = torch.load(f'{emb_dir}/region_label.pt', map_location='cpu')
musk_he_embedding = torch.load(f'{emb_dir}/musk_he_embedding.pt', map_location='cpu')

print(f'HE: {he_embedding.shape}, CODEX: {codex_embedding.shape}, Text: {text_embedding.shape}')
print(f'Region label: {region_label.shape}, unique: {region_label.unique().shape[0]}')
print(f'MUSK HE: {musk_he_embedding.shape}')

# ---- Vocab ----
vocab = pickle.load(open('/data/enable_data/new_individual_samples/final_biomarker_list.pkl', 'rb'))
vocab[vocab == 'PGP9.5'] = 'PGP9_5'
for i in range(len(vocab)):
    if '.' in vocab[i]:
        vocab[i] = vocab[i].replace('.', '_')
vocab_list = list(vocab)
cfg.model.vocab = vocab
print(f'Vocab: {len(vocab_list)} biomarkers')

N = he_embedding.shape[0]
print(f'Total patches: {N}')

In [ ]:
# ============================================================
# Cell 1: Load model, build dataset (TrimodalDatasetViTPickeVerion
#          with return_raw=True, codex_transform=None) to get
#          float32 CODEX and patch/region ordering.
# ============================================================

# ---- ESM embeddings for marker embedding ----
esm_embeddings = {}
esm_dir = '/project/zhihuanglab/common/datasets/enable_data/esm_embeddings'
for pt_file in [f for f in os.listdir(esm_dir) if f.endswith('.pt')]:
    marker = pt_file.replace('.pt', '')
    try:
        esm_embeddings[marker] = torch.load(os.path.join(esm_dir, pt_file), map_location='cpu')
    except Exception:
        pass
known_markers = [m for m in vocab_list if m not in esm_embeddings]
if not esm_embeddings:
    esm_embeddings = {f'marker_{i}': torch.randn(1152) for i in range(10)}

marker_embedding = MarkerEmbedding(
    esm_embeddings, known_markers=known_markers,
    embedding_dim=1152, model_dim=cfg.model.codex_dim
)

tokenizer = BertTokenizer.from_pretrained(cfg.model.text_model)

model = Haiku(
    hf_model=cfg.model.text_model,
    codex_dim=cfg.model.codex_dim, text_dim=cfg.model.text_dim,
    he_dim=cfg.model.he_dim, projection_dim=cfg.model.projection_dim,
    shared_projection=cfg.model.shared_projection,
    marker_embedding=marker_embedding,
    freeze_bert_layers=True, tune_bert_layers=[10, 11],
    freeze_he_encoder=cfg.model.freeze_he_encoder,
    freeze_codex_encoder=cfg.model.freeze_codex_encoder,
    pretrained_weights_path=cfg.model.codex_encoder_weights_path
)
ckpt = torch.load(
    '/home/yancui/Haiku/checkpoints/Trimodal_20260303-0300_full_trainset/clip_checkpoint_epoch_24.pth',
    map_location='cpu'
)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print('Model loaded.')

from torchvision import transforms
from timm.data.constants import IMAGENET_INCEPTION_MEAN, IMAGENET_INCEPTION_STD

he_transform = transforms.Compose([
    transforms.Resize(384, interpolation=3, antialias=True),
    transforms.CenterCrop((384, 384)),
    transforms.Normalize(mean=IMAGENET_INCEPTION_MEAN, std=IMAGENET_INCEPTION_STD),
])

# Use TrimodalDatasetViTPickeVerion with return_raw=True, codex_transform=None
# so that 'codex' is the float32 CODEX image (C, H, W) before grid-reshape
cfg_codex_raw = '/data/enable_data/new_individual_samples'

sample_data = TrimodalDatasetViTPickeVerion(
    cfg_codex_raw, cfg.dataset.he_path, cfg.dataset.text_path,
    ref_ids, tokenizer=tokenizer, max_len=cfg.dataset.max_length,
    he_transform=he_transform, codex_transform=None, return_raw=True
)
text_processing = sample_data.text_processing

print(f'Dataset size: {len(sample_data)}')
print(f'Embedding size: {N}')

# Build patch/region ordering by iterating dataset
# (same order as pre-computed embeddings)
all_patch_ids = []
all_region_ids = []
for idx in range(len(sample_data)):
    rid, pid = sample_data.sample_index[idx]
    all_patch_ids.append(pid)
    all_region_ids.append(rid)

assert len(all_patch_ids) == N, f'Mismatch: dataset {len(all_patch_ids)} vs embeddings {N}'
print(f'Patch ordering verified: {N} patches')

In [ ]:
# ============================================================
# Cell 2: Extract metadata-only text & encode with VLM
# ============================================================

TRANSITION_PATTERNS = [
    r'\s*regarding the molecular profile,.*',
    r'\s*molecularly, the region is characterized by.*',
    r'\s*in terms of protein expression,.*',
    r'\s*quantitative analysis identifies.*',
    r'\s*the spatial-molecular landscape is defined by.*',
]

def extract_metadata_only(full_text):
    """Strip biomarker expression info, keep only metadata/clinical intro."""
    if not full_text or not full_text.strip():
        return ''
    text = full_text.strip()
    for pattern in TRANSITION_PATTERNS:
        text = re.split(pattern, text, flags=re.IGNORECASE)[0]
    return text.strip()

text_dir = cfg.dataset.text_path  # '/data/enable_data/caption_background_text_v3'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

all_meta_texts = []
for pid, rid in zip(all_patch_ids, all_region_ids):
    text_file = os.path.join(text_dir, rid, f'{pid}_backgroud.txt')
    try:
        with open(text_file, 'r') as f:
            full_text = f.read().strip()
        meta_text = extract_metadata_only(full_text)
    except FileNotFoundError:
        meta_text = ''
    all_meta_texts.append(meta_text)

print(f'Loaded {len(all_meta_texts)} metadata texts, non-empty: {sum(1 for t in all_meta_texts if t)}')

batch_size_enc = 64
meta_text_embeddings = []
with torch.no_grad():
    for start in tqdm(range(0, len(all_meta_texts), batch_size_enc), desc='Encoding metadata text'):
        end = min(start + batch_size_enc, len(all_meta_texts))
        batch_texts = all_meta_texts[start:end]
        text_ids_list, att_mask_list = [], []
        for text in batch_texts:
            input_ids, attention_mask = text_processing(text)
            text_ids_list.append(input_ids)
            att_mask_list.append(attention_mask)
        text_batch = {
            'text': torch.stack(text_ids_list).to(device),
            'att_mask': torch.stack(att_mask_list).to(device),
        }
        emb = model.get_features_single_modality(text_batch, modality='text')
        meta_text_embeddings.append(emb.cpu())

meta_text_embedding = torch.cat(meta_text_embeddings, dim=0)
print(f'Metadata text embedding shape: {meta_text_embedding.shape}')

In [ ]:
# ============================================================
# Cell 3: Retrieval — 3 methods: HE→CODEX, Fusion→CODEX, MUSK→CODEX
# ============================================================

@torch.no_grad()
def compute_retrieval_scores(query_emb, gallery_emb, batch_size=2048, device='cuda'):
    """Compute cosine similarity scores between query and gallery."""
    q = F.normalize(query_emb, dim=1).to(device)
    g = F.normalize(gallery_emb, dim=1).to(device)
    Nq = q.shape[0]
    scores_list = []
    for start in range(0, Nq, batch_size):
        end = min(start + batch_size, Nq)
        scores_list.append((q[start:end] @ g.T).cpu())
    return torch.cat(scores_list, dim=0)

@torch.no_grad()
def compute_recall_at_k(scores, match_matrix, top_ks=(1, 5, 10, 20, 50)):
    """Compute Recall@K from score matrix and match matrix."""
    sorted_idx = torch.argsort(scores, dim=1, descending=True)
    sorted_match = torch.gather(match_matrix.float(), 1, sorted_idx)
    num_rel = match_matrix.float().sum(dim=1).clamp(min=1)
    results = {}
    for k in top_ks:
        topk_rel = sorted_match[:, :k].sum(dim=1)
        recall_k = (topk_rel / num_rel).mean().item()
        results[f'R@{k}'] = recall_k
    return results, sorted_idx

# Build match matrix (exact match — same patch)
match_matrix = torch.eye(N, dtype=torch.bool)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Method 1: HE → CODEX
print('Computing HE → CODEX scores...')
s_he = compute_retrieval_scores(he_embedding, codex_embedding, device=device)
recall_he, idx_he = compute_recall_at_k(s_he, match_matrix)
print('HE→CODEX:', {k: f'{v:.4f}' for k, v in recall_he.items()})

# Method 2: Fusion (0.8*HE + 0.2*meta_text) → CODEX
print('\nComputing Fusion (0.8*HE + 0.2*MetaText) → CODEX scores...')
s_meta = compute_retrieval_scores(meta_text_embedding, codex_embedding, device=device)
s_fused = 0.8 * s_he + 0.2 * s_meta
recall_fused, idx_fused = compute_recall_at_k(s_fused, match_matrix)
print('Fusion→CODEX:', {k: f'{v:.4f}' for k, v in recall_fused.items()})

# Method 3: MUSK HE → CODEX
print('\nComputing MUSK HE → CODEX scores...')
s_musk = compute_retrieval_scores(musk_he_embedding, codex_embedding, device=device)
recall_musk, idx_musk = compute_recall_at_k(s_musk, match_matrix)
print('MUSK→CODEX:', {k: f'{v:.4f}' for k, v in recall_musk.items()})

In [ ]:
# ============================================================
# Cell 4: Recall@K bar plot
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family': 'Arial', 'font.size': 12, 'font.weight': 'bold',
    'axes.labelweight': 'bold', 'axes.titleweight': 'bold',
    'svg.fonttype': 'none', 'pdf.fonttype': 42,
})

top_ks = [1, 5, 10, 20, 50]
x = np.arange(len(top_ks))
width = 0.26

he_vals   = [recall_he[f'R@{k}']   for k in top_ks]
fuse_vals = [recall_fused[f'R@{k}'] for k in top_ks]
musk_vals = [recall_musk[f'R@{k}']  for k in top_ks]

fig, ax = plt.subplots(figsize=(10, 6))
bars_he   = ax.bar(x - width, he_vals,   width, label='HE\u2192CODEX',   color='#B7B2D0', edgecolor='black', linewidth=1)
bars_fuse = ax.bar(x,         fuse_vals, width, label='Fusion\u2192CODEX', color='#63C29A', edgecolor='black', linewidth=1)
bars_musk = ax.bar(x + width, musk_vals, width, label='MUSK\u2192CODEX',  color='#90A4AE', edgecolor='black', linewidth=1)

for bars in [bars_he, bars_fuse, bars_musk]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005, f'{h:.3f}',
                ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([str(k) for k in top_ks])
ax.set_xlabel('Top-K', fontsize=14)
ax.set_ylabel('Recall', fontsize=14)
ax.set_title('Recall@K: HE vs Fusion (0.8HE+0.2Meta) vs MUSK', fontsize=14)
ax.legend(fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.tight_layout()

save_dir = '/home/yancui/Haiku/downstream/fusion_pcc_figs'
os.makedirs(save_dir, exist_ok=True)
fig.savefig(os.path.join(save_dir, 'recall_at_k.svg'), bbox_inches='tight', dpi=200)
fig.savefig(os.path.join(save_dir, 'recall_at_k.png'), bbox_inches='tight', dpi=200)
plt.close(fig)
print('Saved recall plot.')

In [ ]:
# ============================================================
# Cell 5: Load raw CODEX expression via dataset (return_raw=True)
#   Use the float32 preprocessed CODEX from pickle files.
#   Compute mean expression per biomarker channel per patch.
# ============================================================

# Build expression matrix: (N, len(vocab_list))
# -1 means missing (biomarker not in that patch's panel)
expression_matrix = -1.0 * np.ones((N, len(vocab_list)), dtype=np.float32)

for i in tqdm(range(N), desc='Loading CODEX expression from dataset'):
    item = sample_data[i]
    codex_raw = item['codex']      # float32 tensor (C, H, W) — raw from pickle
    channels  = item['channels']   # list of biomarker names
    # Mean intensity per channel
    channel_means = codex_raw.mean(dim=(1, 2)).numpy()  # (C,)
    for ci, bm in enumerate(channels):
        bm_clean = bm.replace('.', '_')
        if bm_clean in vocab_list:
            vi = vocab_list.index(bm_clean)
            expression_matrix[i, vi] = channel_means[ci]

expression_matrix_torch = torch.from_numpy(expression_matrix)
print(f'Expression matrix shape: {expression_matrix_torch.shape}')
print(f'Non-missing entries: {(expression_matrix_torch != -1).sum().item()} / {expression_matrix_torch.numel()}')

# Show value range for sanity check
valid_vals = expression_matrix_torch[expression_matrix_torch != -1]
print(f'Expression value range: [{valid_vals.min():.4f}, {valid_vals.max():.4f}], mean: {valid_vals.mean():.4f}')

In [ ]:
# ============================================================
# Cell 6: Filter to shared biomarkers (present in >80% of patches)
#          and z-score normalize per biomarker
# ============================================================

valid_mask = (expression_matrix_torch != -1)  # (N, n_biomarkers)
proportion_present = valid_mask.float().mean(dim=0)  # (n_biomarkers,)
chosen_bm_indices = (proportion_present > 0.8).nonzero(as_tuple=True)[0].tolist()
chosen_biomarkers = [vocab_list[i] for i in chosen_bm_indices]
print(f'Biomarkers with >80% coverage: {len(chosen_biomarkers)}')
print(chosen_biomarkers)

# Filter expression matrix to chosen biomarkers
expr = expression_matrix_torch[:, chosen_bm_indices]  # (N, n_chosen)
expr_valid = (expr != -1)  # (N, n_chosen)

# Per-biomarker normalization (z-score across all valid patches)
expr_norm = expr.clone()
for bi in range(len(chosen_biomarkers)):
    mask = expr_valid[:, bi]
    vals = expr[mask, bi]
    mu = vals.mean()
    sd = vals.std().clamp(min=1e-6)
    expr_norm[mask, bi] = (expr[mask, bi] - mu) / sd

print(f'Normalized expression shape: {expr_norm.shape}')

In [ ]:
# ============================================================
# Cell 7: Per-biomarker PCC via top-5 weighted mean expression
# ============================================================

K = 5

def compute_pcc_per_biomarker(scores, expr_norm, expr_valid, chosen_biomarkers, K=5):
    """Compute per-biomarker PCC using top-K weighted mean expression."""
    Nq = scores.shape[0]
    n_bm = len(chosen_biomarkers)
    sorted_idx = torch.argsort(scores, dim=1, descending=True)
    topk_idx = sorted_idx[:, :K]
    topk_scores = torch.gather(scores, 1, topk_idx)
    topk_weights = F.softmax(topk_scores, dim=1)

    pred_expr = torch.zeros(Nq, n_bm)
    for k_i in range(K):
        gallery_idx = topk_idx[:, k_i]
        gallery_expr = expr_norm[gallery_idx]
        w = topk_weights[:, k_i].unsqueeze(1)
        pred_expr += w * gallery_expr

    pcc_dict = {}
    for bi, bm in enumerate(chosen_biomarkers):
        mask = expr_valid[:, bi]
        true_vals = expr_norm[mask, bi].numpy()
        pred_vals = pred_expr[mask, bi].numpy()
        if len(true_vals) < 3:
            pcc_dict[bm] = np.nan
            continue
        pcc = np.corrcoef(true_vals, pred_vals)[0, 1]
        pcc_dict[bm] = float(pcc) if np.isfinite(pcc) else np.nan
    return pcc_dict

print('Computing per-biomarker PCC for HE...')
pcc_he = compute_pcc_per_biomarker(s_he, expr_norm, expr_valid, chosen_biomarkers, K=K)
print('Computing per-biomarker PCC for Fusion...')
pcc_fused = compute_pcc_per_biomarker(s_fused, expr_norm, expr_valid, chosen_biomarkers, K=K)
print('Computing per-biomarker PCC for MUSK...')
pcc_musk = compute_pcc_per_biomarker(s_musk, expr_norm, expr_valid, chosen_biomarkers, K=K)

print(f'\n{"Biomarker":<20s} {"HE":>8s} {"Fusion":>8s} {"MUSK":>8s}')
print('-' * 48)
for bm in chosen_biomarkers:
    print(f'{bm:<20s} {pcc_he.get(bm, np.nan):8.4f} {pcc_fused.get(bm, np.nan):8.4f} {pcc_musk.get(bm, np.nan):8.4f}')

valid_he = [v for v in pcc_he.values() if np.isfinite(v)]
valid_fused = [v for v in pcc_fused.values() if np.isfinite(v)]
valid_musk = [v for v in pcc_musk.values() if np.isfinite(v)]
print(f'\nMean PCC -- HE: {np.mean(valid_he):.4f}, Fusion: {np.mean(valid_fused):.4f}, MUSK: {np.mean(valid_musk):.4f}')

In [ ]:
# ============================================================
# Cell 8: Grouped barplot — per-biomarker PCC for 3 methods
# ============================================================

from scipy.stats import mannwhitneyu

WIDTH_SCALE = 2.0; BOX_SCALE = 1.2; STYLE_SCALE = 1.0

plt.rcParams.update({
    'svg.fonttype': 'none', 'font.family': 'Arial', 'font.size': 12,
    'font.weight': 'bold', 'axes.labelweight': 'bold',
    'axes.titleweight': 'bold', 'axes.titlesize': 20, 'pdf.fonttype': 42,
    'ytick.labelsize': 20, 'xtick.labelsize': 16,
})

GLOBAL_BAR_WIDTH    = 0.65 * BOX_SCALE
GLOBAL_TRIPLET_STEP = 3.2
GLOBAL_GROUP_GAP    = 2.5
GLOBAL_BRACKET_LW   = 1.40 * STYLE_SCALE
LEFT_MARGIN_IN = 1.05; RIGHT_MARGIN_IN = 0.25; TOP_IN = 1.10; BOTTOM_IN = 1.65
BASE_HEIGHT = 7.0
SUPTITLE_FS = 22; SUPTITLE_Y = 0.995
GROUP_LABEL_Y = 0.93; GROUP_LABEL_FS = 14; GROUP_LABEL_FS_MIN = 11
NARROW_THRESH = 3.0; STAR_FS = 16

marker_groups = {
    "Proliferation /\ncell cycle": ["PCNA", "Ki67"],
    "Immune activation /\nfunction": ["HLA-ABC", "HLA-DR", "HLA-E", "ICOS", "CD40", "IFNg"],
    "Immune exhaustion /\nsuppression": ["PD1", "PDL1", "LAG3", "VISTA", "IDO1", "CD39", "FoxP3"],
    "Cytotoxic /\neffector": ["GranzymeB"],
    "T cell lineage": ["CD3e", "CD4", "CD8", "CD45", "CD45RA", "CD45RO"],
    "Myeloid /\ninnate": ["CD11b", "CD11c", "CD14", "CD141", "CD163", "CD66", "CD68", "MPO", "Gal3"],
    "B cell lineage": ["CD20", "CD21", "CD79", "CD38"],
    "Stromal / vascular /\nECM": ["CD31", "CD34", "CollagenIV", "Caveolin1", "Podoplanin"],
    "Epithelial /\ndifferentiation": ["ECad", "EpCAM", "PanCK", "Keratin8_18", "TP63", "GATA3"],
    "Survival /\nanti-apoptotic": ["BCL2"],
    "Neural /\n other": ["PGP9_5"],
    "Nuclear /\n DNA": ["DAPI"],
}
group_order_full = [
    "Proliferation /\ncell cycle", "Immune activation /\nfunction",
    "Cytotoxic /\neffector", "T cell lineage",
    "Immune exhaustion /\nsuppression", "B cell lineage",
    "Neural /\n other", "Myeloid /\ninnate",
    "Stromal / vascular /\nECM", "Nuclear /\n DNA",
    "Epithelial /\ndifferentiation", "Survival /\nanti-apoptotic",
]
group_bg_colors = {
    "Proliferation /\ncell cycle": "#EEEAF8",
    "Immune activation /\nfunction": "#E9F2FF",
    "Immune exhaustion /\nsuppression": "#FFF0E8",
    "Cytotoxic /\neffector": "#FFE9F2",
    "T cell lineage": "#E9FFF2",
    "Myeloid /\ninnate": "#F1F7E8",
    "B cell lineage": "#F0E8FF",
    "Stromal / vascular /\nECM": "#E8F7F7",
    "Epithelial /\ndifferentiation": "#FFF7E8",
    "Survival /\nanti-apoptotic": "#F7E8E8",
    "Neural /\n other": "#E8E8E8",
    "Nuclear /\n DNA": "#F2F2F2",
}

mcol = {'HE': '#B7B2D0', 'Fusion': '#63C29A', 'MUSK': '#90A4AE'}

marker_groups_filtered = {}
for g, markers in marker_groups.items():
    filtered = [m for m in markers if m in chosen_biomarkers]
    if filtered:
        marker_groups_filtered[g] = filtered

def compute_positions_and_spans(group_order, mg):
    positions = []; xticks = []; xticklabels = []; group_spans = []; triplet_pos = {}
    cursor = 0.0
    nonempty = [g for g in group_order if g in mg and mg[g]]
    for gi, g in enumerate(nonempty):
        markers = mg[g]; x_start = cursor
        for mi, m in enumerate(markers):
            p0, p1, p2 = cursor, cursor + 1.0, cursor + 2.0
            positions.extend([p0, p1, p2])
            xticks.append(cursor + 1.0); xticklabels.append(m)
            triplet_pos[m] = (p0, p1, p2)
            cursor += GLOBAL_TRIPLET_STEP if mi < len(markers) - 1 else 2.0
        group_spans.append((g, x_start, cursor))
        if gi < len(nonempty) - 1:
            cursor += GLOBAL_GROUP_GAP
    if not positions:
        return None
    xmin = min(positions) - 1.0; xmax = max(positions) + 1.0
    return {'positions': positions, 'xticks': xticks, 'xticklabels': xticklabels,
            'group_spans': group_spans, 'triplet_pos': triplet_pos,
            'xmin': xmin, 'xmax': xmax, 'x_span': xmax - xmin}

def apply_fixed_margins(fig, fw, fh):
    fig.subplots_adjust(
        left=min(0.90, max(0.01, LEFT_MARGIN_IN / fw)),
        right=min(0.99, max(0.10, 1.0 - RIGHT_MARGIN_IN / fw)),
        bottom=min(0.85, max(0.01, BOTTOM_IN / fh)),
        top=min(0.99, max(0.10, 1.0 - TOP_IN / fh)))

full_geom = compute_positions_and_spans(group_order_full, marker_groups_filtered)
margins_in = LEFT_MARGIN_IN + RIGHT_MARGIN_IN
INCH_PER_X = (30.0 - margins_in) / max(1e-6, full_geom['x_span']) * WIDTH_SCALE

def p_to_stars(p):
    if not np.isfinite(p): return ''
    if p < 1e-3: return '***'
    if p < 1e-2: return '**'
    if p < 5e-2: return '*'
    return ''

def add_sig_bracket(ax, x1, x2, y, h, text, fontsize=14, lw=1.4):
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=lw, c='k', zorder=4)
    ax.text((x1 + x2) / 2, y + h, text, ha='center', va='bottom',
            fontsize=fontsize, fontweight='bold', zorder=5)

def plot_pcc_grouped(group_order, mg, pcc_he, pcc_fused, pcc_musk, out_path, title=None):
    geom = compute_positions_and_spans(group_order, mg)
    if geom is None:
        print('No data'); return
    pos = geom['positions']; xt = geom['xticks']; xtl = geom['xticklabels']
    gs = geom['group_spans']; tp = geom['triplet_pos']
    xmin, xmax, xsp = geom['xmin'], geom['xmax'], geom['x_span']

    fw = margins_in + INCH_PER_X * xsp
    fh = BASE_HEIGHT
    fig, ax = plt.subplots(1, 1, figsize=(fw, fh))
    apply_fixed_margins(fig, fw, fh)
    if title:
        fig.suptitle(title, fontsize=SUPTITLE_FS, fontweight='bold', y=SUPTITLE_Y)

    bar_w = GLOBAL_BAR_WIDTH

    for g in [g for g in group_order if g in mg and mg[g]]:
        for m in mg[g]:
            p0, p1, p2 = tp[m]
            v_he = pcc_he.get(m, np.nan)
            v_fu = pcc_fused.get(m, np.nan)
            v_mu = pcc_musk.get(m, np.nan)
            ax.bar(p0, v_he if np.isfinite(v_he) else 0, bar_w,
                   color=mcol['HE'], edgecolor='black', linewidth=0.5, zorder=2)
            ax.bar(p1, v_fu if np.isfinite(v_fu) else 0, bar_w,
                   color=mcol['Fusion'], edgecolor='black', linewidth=0.5, zorder=2)
            ax.bar(p2, v_mu if np.isfinite(v_mu) else 0, bar_w,
                   color=mcol['MUSK'], edgecolor='black', linewidth=0.5, zorder=2)

    for (g, x0, x1_) in gs:
        ax.axvspan(x0 - 0.6, x1_ + 0.6, alpha=0.35,
                   color=group_bg_colors.get(g, '#F5F5F5'), zorder=0)

    ax.set_xticks(xt)
    ax.set_xticklabels(xtl, rotation=75, ha='center', fontsize=16)
    ax.set_ylabel('Pearson Correlation', fontsize=24)

    for (g, x0, x1_) in gs:
        fs = GROUP_LABEL_FS if (x1_ - x0) >= NARROW_THRESH else max(GROUP_LABEL_FS_MIN, int(GROUP_LABEL_FS * 0.85))
        ax.text((x0 + x1_) / 2, GROUP_LABEL_Y, g, ha='center', va='top',
                transform=ax.get_xaxis_transform(), fontsize=fs, fontweight='bold')
    for (_, _, x1_) in gs[:-1]:
        ax.axvline(x1_ + 1.15, ymin=0.06, ymax=0.92, lw=1.2, alpha=0.25, color='k')

    ax.set_xlim(xmin, xmax)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor=mcol[k], label=k) for k in mcol],
              loc='upper right', fontsize=12)

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    fig.savefig(out_path, pad_inches=0, bbox_inches='tight', dpi=200)
    fig.savefig(out_path.replace('.svg', '.png'), pad_inches=0, bbox_inches='tight', dpi=200)
    plt.close(fig)
    print(f'Saved: {out_path}')

save_root = '/home/yancui/Haiku/downstream/fusion_pcc_figs'

plot_pcc_grouped(
    group_order_full, marker_groups_filtered,
    pcc_he, pcc_fused, pcc_musk,
    os.path.join(save_root, 'biomarker_pcc_grouped_all.svg'),
    title='Per-Biomarker PCC: HE vs Fusion vs MUSK'
)

big_group_map = {
    'Tumor Intrinsic Programs': ['Proliferation /\ncell cycle', 'Nuclear /\n DNA',
                                  'Epithelial /\ndifferentiation', 'Survival /\nanti-apoptotic'],
    'Adaptive Immune Compartment': ['Immune activation /\nfunction', 'Cytotoxic /\neffector',
                                    'T cell lineage', 'Immune exhaustion /\nsuppression', 'B cell lineage'],
    'Innate & Myeloid Compartment': ['Myeloid /\ninnate'],
    'Stromal / Microenvironment Programs': ['Stromal / vascular /\nECM'],
    'Neural / Other Programs': ['Neural /\n other'],
}

def ordered_subgroups(subs):
    s = set(subs)
    return [g for g in group_order_full if g in s]

save_big = os.path.join(save_root, 'big_groups')
for bn, fl in big_group_map.items():
    fg = ordered_subgroups(fl)
    fname = bn.lower().replace('&', 'and').replace('/', '_').replace(' ', '_')
    plot_pcc_grouped(
        fg, marker_groups_filtered,
        pcc_he, pcc_fused, pcc_musk,
        os.path.join(save_big, f'pcc_{fname}.svg'),
        title=f'{bn} (PCC)'
    )
print('Done with grouped barplots.')

In [ ]:
# ============================================================
# Cell 9: Per-region PCC for violin / boxplot grouped by function
# ============================================================

region_label_np = region_label.cpu().numpy().squeeze()  # (N,)

def compute_per_region_pcc(scores, expr_norm, expr_valid, chosen_biomarkers, region_labels, K=5):
    sorted_idx = torch.argsort(scores, dim=1, descending=True)
    topk_idx = sorted_idx[:, :K]
    topk_scores = torch.gather(scores, 1, topk_idx)
    topk_weights = F.softmax(topk_scores, dim=1)
    n_bm = len(chosen_biomarkers)
    Nq = scores.shape[0]

    pred_expr = torch.zeros(Nq, n_bm)
    for k_i in range(K):
        gallery_idx = topk_idx[:, k_i]
        gallery_expr = expr_norm[gallery_idx]
        w = topk_weights[:, k_i].unsqueeze(1)
        pred_expr += w * gallery_expr

    region_pcc = {}
    for r in np.unique(region_labels):
        rmask = (region_labels == r)
        region_pcc[int(r)] = {}
        for bi, bm in enumerate(chosen_biomarkers):
            bm_mask = expr_valid[:, bi].numpy()
            mask = rmask & bm_mask
            if mask.sum() < 5:
                continue
            true_vals = expr_norm[mask, bi].numpy()
            pred_vals = pred_expr[mask, bi].numpy()
            if np.std(true_vals) < 1e-8 or np.std(pred_vals) < 1e-8:
                continue
            pcc = np.corrcoef(true_vals, pred_vals)[0, 1]
            if np.isfinite(pcc):
                region_pcc[int(r)][bm] = float(pcc)
    return region_pcc

print('Computing per-region PCC for HE...')
rpcc_he = compute_per_region_pcc(s_he, expr_norm, expr_valid, chosen_biomarkers, region_label_np, K=K)
print('Computing per-region PCC for Fusion...')
rpcc_fused = compute_per_region_pcc(s_fused, expr_norm, expr_valid, chosen_biomarkers, region_label_np, K=K)
print('Computing per-region PCC for MUSK...')
rpcc_musk = compute_per_region_pcc(s_musk, expr_norm, expr_valid, chosen_biomarkers, region_label_np, K=K)
print(f'Regions with data -- HE: {len(rpcc_he)}, Fusion: {len(rpcc_fused)}, MUSK: {len(rpcc_musk)}')

In [ ]:
# ============================================================
# Cell 10: Grouped boxplot (violin-style) with per-region PCC values
# ============================================================

GLOBAL_BOX_WIDTH    = 0.65 * BOX_SCALE
GLOBAL_JITTER_WIDTH = 0.14 * BOX_SCALE
GLOBAL_POINT_SIZE   = 28.0 * (STYLE_SCALE ** 2)

def gather_pcc_values(rpcc, bm):
    return np.array([rpcc[r][bm] for r in rpcc if bm in rpcc[r]], dtype=float)

def plot_pcc_boxplot_grouped(group_order, mg, rpcc_he, rpcc_fused, rpcc_musk, out_path, title=None):
    geom = compute_positions_and_spans(group_order, mg)
    if geom is None:
        print('No data'); return
    pos = geom['positions']; xt = geom['xticks']; xtl = geom['xticklabels']
    gs = geom['group_spans']; tp = geom['triplet_pos']
    xmin, xmax, xsp = geom['xmin'], geom['xmax'], geom['x_span']

    fw = margins_in + INCH_PER_X * xsp
    fh = BASE_HEIGHT
    fig, ax = plt.subplots(1, 1, figsize=(fw, fh))
    apply_fixed_margins(fig, fw, fh)
    if title:
        fig.suptitle(title, fontsize=SUPTITLE_FS, fontweight='bold', y=SUPTITLE_Y)

    bd = []; bc = []; mv = {}; mp = {}
    for g in [g for g in group_order if g in mg and mg[g]]:
        for m in mg[g]:
            vh = gather_pcc_values(rpcc_he, m).tolist()
            vf = gather_pcc_values(rpcc_fused, m).tolist()
            vm = gather_pcc_values(rpcc_musk, m).tolist()
            bd.append(vh); bc.append(mcol['HE'])
            bd.append(vf); bc.append(mcol['Fusion'])
            bd.append(vm); bc.append(mcol['MUSK'])
            mv[m] = (vh, vf, vm)
            p_u = np.nan
            if len(vf) >= 2 and len(vm) >= 2:
                try:
                    p_u = mannwhitneyu(vf, vm, alternative='greater').pvalue
                except Exception:
                    pass
            mp[m] = float(p_u) if np.isfinite(p_u) else np.nan

    for (g, x0, x1_) in gs:
        ax.axvspan(x0 - 0.6, x1_ + 0.6, alpha=0.35,
                   color=group_bg_colors.get(g, '#F5F5F5'), zorder=0)

    bp = ax.boxplot(bd, positions=pos, widths=GLOBAL_BOX_WIDTH, patch_artist=True, showmeans=False)
    for p, c in zip(bp['boxes'], bc):
        p.set_facecolor(c); p.set_alpha(0.75)
    for p_, v, c in zip(pos, bd, bc):
        if not v: continue
        jx = [p_ + np.random.uniform(-GLOBAL_JITTER_WIDTH, GLOBAL_JITTER_WIDTH) for _ in v]
        ax.scatter(jx, v, alpha=0.55, color=c, s=GLOBAL_POINT_SIZE,
                   edgecolor='k', linewidth=0.3, zorder=3)

    ax.set_xticks(xt)
    ax.set_xticklabels(xtl, rotation=75, ha='center', fontsize=16)
    ax.set_ylabel('Pearson Correlation (per region)', fontsize=24)

    for (g, x0, x1_) in gs:
        fs = GROUP_LABEL_FS if (x1_ - x0) >= NARROW_THRESH else max(GROUP_LABEL_FS_MIN, int(GROUP_LABEL_FS * 0.85))
        ax.text((x0 + x1_) / 2, GROUP_LABEL_Y, g, ha='center', va='top',
                transform=ax.get_xaxis_transform(), fontsize=fs, fontweight='bold')
    for (_, _, x1_) in gs[:-1]:
        ax.axvline(x1_ + 1.15, ymin=0.06, ymax=0.92, lw=1.2, alpha=0.25, color='k')

    ymin_, ymax_ = ax.get_ylim(); yrng = max(1e-6, ymax_ - ymin_)
    h = 0.015 * yrng; pad = 0.02 * yrng
    for m, (p0, p1, p2) in tp.items():
        stars = p_to_stars(mp.get(m, np.nan))
        if not stars: continue
        allv = sum((list(x) for x in mv[m] if x), [])
        if not allv: continue
        add_sig_bracket(ax, p1, p2, max(allv) + pad, h, stars, STAR_FS, GLOBAL_BRACKET_LW)

    ax.set_ylim(ymin_, ax.get_ylim()[1] + 0.06 * yrng)
    ax.set_xlim(xmin, xmax)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor=mcol[k], label=k) for k in mcol],
              loc='upper right', fontsize=12)

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    fig.savefig(out_path, pad_inches=0, bbox_inches='tight', dpi=200)
    fig.savefig(out_path.replace('.svg', '.png'), pad_inches=0, bbox_inches='tight', dpi=200)
    plt.close(fig)
    print(f'Saved: {out_path}')

plot_pcc_boxplot_grouped(
    group_order_full, marker_groups_filtered,
    rpcc_he, rpcc_fused, rpcc_musk,
    os.path.join(save_root, 'biomarker_pcc_boxplot_grouped_all.svg'),
    title='Per-Region Biomarker PCC: HE vs Fusion vs MUSK'
)

for bn, fl in big_group_map.items():
    fg = ordered_subgroups(fl)
    fname = bn.lower().replace('&', 'and').replace('/', '_').replace(' ', '_')
    plot_pcc_boxplot_grouped(
        fg, marker_groups_filtered,
        rpcc_he, rpcc_fused, rpcc_musk,
        os.path.join(save_big, f'pcc_boxplot_{fname}.svg'),
        title=f'{bn} (per-region PCC)'
    )
print('Done with grouped boxplots.')

In [ ]:
# ============================================================
# Cell 11: Aggregate summary — mean PCC + SEM bar chart
# ============================================================

methods = ['HE', 'Fusion', 'MUSK']
pcc_dicts = [pcc_he, pcc_fused, pcc_musk]
colors = [mcol[m] for m in methods]

means = []
sems = []
for pd_ in pcc_dicts:
    vals = [v for v in pd_.values() if np.isfinite(v)]
    means.append(np.mean(vals))
    sems.append(np.std(vals) / np.sqrt(len(vals)))

fig, ax = plt.subplots(figsize=(5, 5))
x = np.arange(len(methods))
bars = ax.bar(x, means, yerr=sems, capsize=5, color=colors,
              edgecolor='black', linewidth=1, width=0.6)

for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{m:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=14)
ax.set_ylabel('Mean PCC', fontsize=16)
ax.set_title('Aggregate Pearson Correlation', fontsize=16, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.tight_layout()

fig.savefig(os.path.join(save_root, 'aggregate_pcc.svg'), bbox_inches='tight', dpi=200)
fig.savefig(os.path.join(save_root, 'aggregate_pcc.png'), bbox_inches='tight', dpi=200)
plt.close(fig)
print('Done.')

# Save all results
results = {
    'recall': {'HE': recall_he, 'Fusion': recall_fused, 'MUSK': recall_musk},
    'pcc_global': {'HE': pcc_he, 'Fusion': pcc_fused, 'MUSK': pcc_musk},
    'pcc_per_region': {'HE': rpcc_he, 'Fusion': rpcc_fused, 'MUSK': rpcc_musk},
    'chosen_biomarkers': chosen_biomarkers,
}
with open(os.path.join(save_root, 'fusion_pcc_results.pkl'), 'wb') as f:
    pickle.dump(results, f)
print('Results saved to fusion_pcc_results.pkl')